In [3]:
import pandas as pd

# Define the path to the parquet file
# Ensure 'artgraph_metadata.parquet' is in the same directory as your notebook
METADATA_FILE_PATH = 'artgraph_metadata.parquet'

print(f"Attempting to load metadata file: {METADATA_FILE_PATH}")

try:
    # Load the parquet file into a Pandas DataFrame
    metadata_df = pd.read_parquet(METADATA_FILE_PATH)
    
    print("\nMetadata loaded successfully!")
    
    # Print the first 3 rows of the DataFrame
    print("\n--- First 3 rows of metadata ---")
    print(metadata_df.head(3))
    
    # Print the names of all available columns
    print("\n--- Available columns in metadata ---")
    print(metadata_df.columns.tolist())
    
    # Print the total number of rows and columns
    print(f"\nDataFrame dimensions: {metadata_df.shape[0]} rows, {metadata_df.shape[1]} columns.")
    
except FileNotFoundError:
    print(f"\nERROR: The file '{METADATA_FILE_PATH}' was not found.")
    print("Please ensure the file is in the same directory as your notebook.")
except Exception as e:
    print(f"\nAn unexpected error occurred while loading metadata: {e}")

print("\nMetadata exploration complete.")

Attempting to load metadata file: artgraph_metadata.parquet

Metadata loaded successfully!

--- First 3 rows of metadata ---
                        ArtworkTitle            ArtistName ArtworkYear Period  \
0  Menton.&#160;Beach with umbrellas  zinaida-serebriakova        1931   None   
1                        Autumn Song                  erte        None   None   
2                           Feathers                  erte        None   None   

      Style                                           FileName  \
0  art deco  zinaida-serebriakova_menton-beach-with-umbrell...   
1  art deco                               erte_autumn-song.jpg   
2  art deco                                  erte_feathers.jpg   

               Genre                                           Movement  
0     genre painting  Mir iskusstva, Neoclassical architecture, Repr...  
1  symbolic painting                                               None  
2             design                                           

In [5]:
import os
import base64
from pathlib import Path
from PIL import Image
import pandas as pd
from ollama import chat
from io import BytesIO
import re
import numpy as np
import torch
import csv

# --- GLOBAL CONFIGURATION ---
# Path to the image directory
# Ensure 'images100' is the main folder containing your images.
IMAGE_SOURCE_DIR = './images100' 

# Ollama model to use
OLLAMA_MODEL = 'qwen2.5vl'

# Limit processing to the first N images (set to None to process all images)
IMAGES_TO_PROCESS = None # Set to None to process all images

# Number of fragments for the 2x2 grid
GRID_ROWS = 2
GRID_COLS = 2
TOTAL_GRID_SEGMENTS = GRID_ROWS * GRID_COLS # Will be 4

# Maximum number of tokens to generate for descriptions (adjust manually for your tests)
MAX_TOKENS = 512 

# Path to the metadata file
METADATA_FILE_PATH = 'artgraph_metadata.parquet'

# Output CSV filename for Cell 2
MAIN_OUTPUT_CSV_FILENAME = f"image_analysis_qwen_2x2_with_metadata_max_tokens_{MAX_TOKENS}.csv"

# Temperature for model generation (higher values = more creativity, lower values = more deterministic)
TEMPERATURE = 0.7 

# --- PROMPTS ADAPTED FOR IMAGES (Requesting English output and full detail) ---
# This prompt is designed to give absolute priority to the fragment description,
# using metadata as secondary informative context, if available.
PROMPT_FRAGMENT_BASE = (
    "Describe comprehensively and with rich detail **only what is visible in this specific portion of the image.** "
    "Elaborate extensively on the elements, subjects, colors, textures, shapes, lighting, and composition present within the fragment. "
    "Do not make inferences or add details not directly observable here. "
    "Provide a complete and in-depth description in English." # Emphasized completeness and in-depth
)
# Added a variable for metadata context that will be appended to the base prompt
METADATA_CONTEXT_SUFFIX = (
    " (Context: this portion is part of {metadata_info})."
)

# New prompt for overall image analysis
PROMPT_TOTAL_IMAGE_BASE = (
    "Analyze and describe the entire image as a whole with maximum detail and completeness. "
    "Focus extensively on general themes, overall composition, the intricate interaction between elements, "
    "and the overarching message, mood, or emotion the artwork conveys. "
    "Provide a holistic and exhaustive description, going beyond individual fragment details. "
    "Provide your description in English." # Emphasized exhaustive and holistic
)


# --- UTILITY FUNCTIONS ---

def encode_image_to_base64(image: Image.Image, size=(256, 256)) -> str:
    """
    Resizes a PIL image and encodes it to base64.
    """
    try:
        img_resized = image.resize(size) 
        
        buffer = BytesIO()
        img_resized.save(buffer, format="JPEG") 
        img_bytes = buffer.getvalue()
        
        return base64.b64encode(img_bytes).decode('utf-8')
    except Exception as e:
        print(f"Error encoding image: {e}")
        return None

def split_image_into_grid_segments(image: Image.Image, rows: int, cols: int) -> list[Image.Image]:
    """
    Splits a PIL image into a grid of segments (e.g., 2x2).
    Returns segments in reading order (left to right, top to bottom).
    """
    width, height = image.size
    segment_width = width // cols
    segment_height = height // rows
    segments = []
    
    for r in range(rows):
        for c in range(cols):
            left = c * segment_width
            upper = r * segment_height
            right = (c + 1) * segment_width if c < cols - 1 else width
            lower = (r + 1) * segment_height if r < rows - 1 else height
            
            segment = image.crop((left, upper, right, lower))
            segments.append(segment)
            
    return segments

def generate_description_ollama(model_name: str, image_base64: str, prompt: str) -> str:
    """
    Generates a description using a specific Ollama model with a base64 image.
    """
    try:
        response = chat(
            model=model_name,
            messages=[
                {
                    'role': 'user',
                    'content': prompt,
                    'images': [image_base64],
                }
            ],
            options={
                'num_predict': MAX_TOKENS,
                'temperature': TEMPERATURE # TEMPERATURE is now globally defined
            }
        )
        return response.message.content.strip()
    except Exception as e:
        return f"Error during inference with {model_name}: {e}"

# --- IMAGE DIRECTORY CHECK ---
if not os.path.isdir(IMAGE_SOURCE_DIR):
    print(f"ERROR: The image directory '{IMAGE_SOURCE_DIR}' was not found.")
    print("Please ensure the 'images100' folder is in the same directory as your notebook and contains your images.")

print("Setup complete and utility functions loaded.")

# Cell 3: Image Description Generation with Ollama and Metadata

print(f"\n\n--- STARTING IMAGE ANALYSIS with {OLLAMA_MODEL} (via Ollama) ---")
print(f" MAX_TOKENS set to: {MAX_TOKENS}")

# This list will collect dictionaries for each row of the final CSV
all_image_descriptions_data = []
processed_images_count = 0

# --- Metadata loading and preparation ---
metadata_df = None
metadata_dict = {} # Dictionary for fast lookup
try:
    # Use METADATA_FILE_PATH as defined in Cell 2
    metadata_df = pd.read_parquet(METADATA_FILE_PATH, engine='pyarrow') 
    print(f"\nMetadata '{METADATA_FILE_PATH}' loaded successfully.")
    
    # INITIAL AND AGGRESSIVE CLEANING OF NEWLY LOADED METADATA HERE
    # Apply cleaning to all 'object' type (string) columns
    for col in metadata_df.select_dtypes(include=['object']).columns:
        # Replace newlines, carriage returns, and &#160; with spaces, then strip extra whitespace
        metadata_df[col] = metadata_df[col].astype(str).str.replace('\n', ' ').str.replace('\r', ' ').str.replace('&#160;', ' ').str.strip()
        # Remove any double spaces for further cleaning
        metadata_df[col] = metadata_df[col].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

    # Prepare a dictionary for efficient lookup using 'FileName' as the key
    metadata_dict = metadata_df.set_index('FileName').to_dict('index')
    print("Metadata prepared for quick lookup by 'FileName' and cleaned of special characters.")

except FileNotFoundError:
    print(f"\nWARNING: File '{METADATA_FILE_PATH}' not found. Descriptions will not be enriched with metadata.")
    metadata_df = None
except Exception as e:
    print(f"\nERROR loading/preparing metadata: {e}. Descriptions will not be enriched with metadata.")
    metadata_df = None
# --- END Metadata Loading ---

# Get the list of image files from the image directory (IMAGE_SOURCE_DIR defined in Cell 2)
all_image_files_in_dir = [f for f in os.listdir(IMAGE_SOURCE_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tiff"))]

# Filter images based on available metadata and apply the IMAGES_TO_PROCESS limit
image_files_to_process = sorted([f for f in all_image_files_in_dir if f in metadata_dict]) 

if IMAGES_TO_PROCESS is not None and IMAGES_TO_PROCESS > 0:
    image_files_to_process = image_files_to_process[:IMAGES_TO_PROCESS]
    print(f"\nLimited to processing the first {IMAGES_TO_PROCESS} valid images.")
else:
    print(f"\nProcessing all {len(image_files_to_process)} valid images in folder '{IMAGE_SOURCE_DIR}'.")


if not image_files_to_process:
    print("\nATTENTION: No images found in the specified directory that have a corresponding entry in the metadata. Ensure file names match between the folder and the 'FileName' column of the parquet file.")
    print("Process terminated with no images to process.")
else:
    for filename in image_files_to_process: 
        image_file_path = os.path.join(IMAGE_SOURCE_DIR, filename)
        print(f"\n--- PROCESSING IMAGE: {filename} ({processed_images_count + 1}{f'/{len(image_files_to_process)}' if IMAGES_TO_PROCESS is not None else ''}) ---")

        # Retrieve metadata for the current image
        image_metadata = metadata_dict.get(filename, {})

        # Extract metadata values (already cleaned upon parquet loading)
        artwork_title = image_metadata.get('ArtworkTitle', '')
        artist_name = image_metadata.get('ArtistName', '')
        artwork_year = image_metadata.get('ArtworkYear', '')
        period = image_metadata.get('Period', '')
        style = image_metadata.get('Style', '')
        genre = image_metadata.get('Genre', '')
        movement = image_metadata.get('Movement', '')

        try:
            original_image = Image.open(image_file_path).convert("RGB")
            print(f"Original image loaded: {filename} (dim: {original_image.size})")

            # --- Building metadata context for prompts ---
            metadata_info_str = ""
            metadata_parts_list = [] 
            
            # Add parts only if the value is not an empty string or 'None' textually (after cleaning)
            if artwork_title and artwork_title.lower() not in ['none', 'unknown']:
                metadata_parts_list.append(f"the artwork '{artwork_title}'")
            if artwork_year and artwork_year.lower() not in ['none', 'unknown']:
                metadata_parts_list.append(f"created in {artwork_year}")
            if artist_name and artist_name.lower() not in ['none', 'unknown']:
                metadata_parts_list.append(f"by artist {artist_name}")
            if style and style.lower() not in ['none', 'unknown']:
                metadata_parts_list.append(f"in the style of {style}")
            if genre and genre.lower() not in ['none', 'unknown']:
                metadata_parts_list.append(f"and belonging to the genre {genre}")
            if movement and movement.lower() not in ['none', 'unknown']:
                metadata_parts_list.append(f"with the movement {movement}")

            if metadata_parts_list:
                metadata_info_str = ", ".join(metadata_parts_list)
                print(f" Metadata context generated for '{filename}': '{metadata_info_str}'")
            
            # Combine the base prompt with the metadata context (PROMPT_FRAGMENT_BASE, PROMPT_TOTAL_IMAGE_BASE, METADATA_CONTEXT_SUFFIX defined in Cell 2)
            final_fragment_prompt = PROMPT_FRAGMENT_BASE
            if metadata_info_str:
                final_fragment_prompt += METADATA_CONTEXT_SUFFIX.format(metadata_info=metadata_info_str)
            
            final_total_prompt = PROMPT_TOTAL_IMAGE_BASE
            if metadata_info_str:
                final_total_prompt += METADATA_CONTEXT_SUFFIX.format(metadata_info=metadata_info_str)


            print(f" Final prompt for fragments: {final_fragment_prompt[:100]}...")
            print(f" Final prompt for total analysis: {final_total_prompt[:100]}...")
            # --- END Prompt Construction ---

            # Fragment into 2x2 and describe each segment (GRID_ROWS, GRID_COLS defined in Cell 2)
            segments = split_image_into_grid_segments(original_image, rows=GRID_ROWS, cols=GRID_COLS)
            print(f" Image divided into {len(segments)} segments ({GRID_ROWS}x{GRID_COLS} grid).")

            # Add fragment descriptions
            for i, segment in enumerate(segments):
                print(f"   Generating description for segment {i+1}...")
                base64_segment = encode_image_to_base64(segment, size=(256, 256)) # encode_image_to_base64 defined in Cell 2
                
                fragment_description = ""
                if base64_segment:
                    fragment_description = generate_description_ollama(OLLAMA_MODEL, base64_segment, final_fragment_prompt)
                    # Apply cleaning to the generated model description
                    # Remove newlines, carriage returns, and then clean double spaces
                    fragment_description = fragment_description.replace('\n', ' ').replace('\r', ' ').strip()
                    fragment_description = re.sub(r'\s+', ' ', fragment_description).strip() # Remove double spaces

                    print(f"   Segment {i+1} Desc: {fragment_description[:100]}...")
                else:
                    fragment_description = f"Error: Segment {i+1} encoding failed."

                # Add the fragment row to the data list for the CSV
                row = {
                    "Image Name": filename if i == 0 else '', # Translated "Nome Immagine" to "Image Name"
                    "ArtworkTitle": artwork_title if i == 0 else '',
                    "ArtistName": artist_name if i == 0 else '',
                    "ArtworkYear": artwork_year if i == 0 else '',
                    "Period": period if i == 0 else '',
                    "Style": style if i == 0 else '',
                    "Genre": genre if i == 0 else '',
                    "Movement": movement if i == 0 else '',
                    "Description Type": f"Fragment {i+1}", # Translated "Tipo Descrizione" to "Description Type"
                    "Description": fragment_description # Translated "Descrizione" to "Description"
                }
                all_image_descriptions_data.append(row)
            
            # Generate description for the entire image
            print("  Generating description for the entire image...")
            base64_full_image = encode_image_to_base64(original_image, size=(512, 512))
            
            full_image_description = ""
            if base64_full_image:
                full_image_description = generate_description_ollama(OLLAMA_MODEL, base64_full_image, final_total_prompt)
                # Apply cleaning to the generated model description
                # Remove newlines, carriage returns, and then clean double spaces
                full_image_description = full_image_description.replace('\n', ' ').replace('\r', ' ').strip()
                full_image_description = re.sub(r'\s+', ' ', full_image_description).strip() # Remove double spaces

                print(f"  Total Description: {full_image_description[:100]}...")
            else:
                full_image_description = "Error: Full image encoding failed."

            # Add the total description row to the data list for the CSV
            row_total = {
                "Image Name": '', # Translated "Nome Immagine" to "Image Name"
                "ArtworkTitle": '', "ArtistName": '', "ArtworkYear": '', "Period": '', "Style": '', "Genre": '', "Movement": '',
                "Description Type": "Total", # Translated "Tipo Descrizione" to "Description Type"
                "Description": full_image_description # Translated "Descrizione" to "Description"
            }
            all_image_descriptions_data.append(row_total)
            
        except Exception as e:
            print(f"CRITICAL ERROR during image processing {filename}: {e}")
            # Add error rows for fragments and total if an error occurs
            for i in range(TOTAL_GRID_SEGMENTS): # TOTAL_GRID_SEGMENTS defined in Cell 2
                error_row_fragment = {
                    "Image Name": filename if i == 0 else '',
                    "ArtworkTitle": artwork_title if i == 0 else '',
                    "ArtistName": artist_name if i == 0 else '',
                    "ArtworkYear": artwork_year if i == 0 else '',
                    "Period": period if i == 0 else '',
                    "Style": style if i == 0 else '',
                    "Genre": genre if i == 0 else '',
                    "Movement": movement if i == 0 else '',
                    "Description Type": f"Fragment {i+1}",
                    "Description": f"ERROR DURING PROCESSING: {e}"
                }
                all_image_descriptions_data.append(error_row_fragment)
            
            error_row_total = {
                "Image Name": '',
                "ArtworkTitle": '', "ArtistName": '', "ArtworkYear": '', "Period": '', "Style": '', "Genre": '', "Movement": '',
                "Description Type": "Total",
                "Description": f"ERROR DURING PROCESSING: {e}"
            }
            all_image_descriptions_data.append(error_row_total)

        processed_images_count += 1
        
        # --- Optimization: Clear CUDA cache after each image ---
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print(f" CUDA cache cleared after processing {filename}.")

    # --- AFTER THE MAIN LOOP, WHEN ALL IMAGES HAVE BEEN PROCESSED ---

    # Define the column order for the final DataFrame (crucial for format)
    column_order = [
        "Image Name", # Translated "Nome Immagine" to "Image Name"
        "ArtworkTitle",
        "ArtistName",
        "ArtworkYear",
        "Period",
        "Style",
        "Genre",
        "Movement",
        "Description Type", # Translated "Tipo Descrizione" to "Description Type"
        "Description" # Translated "Descrizione" to "Description"
    ]

    # Create the final DataFrame from the collected data
    df_output_images = pd.DataFrame(all_image_descriptions_data, columns=column_order)

    print(f"\nImage analysis process completed for the first {processed_images_count} images with {OLLAMA_MODEL}.")
    print("Pandas DataFrame 'df_output_images' successfully created.")

    print("\n--- Preview of DataFrame 'df_output_images' ---")
    # Print the first 10 rows to show an example of fragments and total analysis for one or more images
    print(df_output_images.head(10).to_markdown(index=False)) 

    # Save the DataFrame to a CSV file WITHOUT the 'quoting=' clause
    # This will use Pandas' default quoting behavior (QUOTE_MINIMAL or equivalent)
    df_output_images.to_csv(MAIN_OUTPUT_CSV_FILENAME, index=False, encoding='utf-8')
    print(f"\nDataFrame saved as '{MAIN_OUTPUT_CSV_FILENAME}' in the same directory as the notebook with default (minimal) quoting.")

print("\nFull process terminated.")

Setup complete and utility functions loaded.


--- STARTING IMAGE ANALYSIS with qwen2.5vl (via Ollama) ---
 MAX_TOKENS set to: 512

Metadata 'artgraph_metadata.parquet' loaded successfully.
Metadata prepared for quick lookup by 'FileName' and cleaned of special characters.

Processing all 100 valid images in folder './images100'.

--- PROCESSING IMAGE: aleksey-savrasov_courtyard-spring-1853.jpg (1) ---
Original image loaded: aleksey-savrasov_courtyard-spring-1853.jpg (dim: (220, 275))
 Metadata context generated for 'aleksey-savrasov_courtyard-spring-1853.jpg': 'the artwork 'Courtyard. Spring.', created in 1853, by artist aleksey-savrasov, in the style of realism, and belonging to the genre cityscape, with the movement Realism (arts)'
 Final prompt for fragments: Describe comprehensively and with rich detail **only what is visible in this specific portion of the...
 Final prompt for total analysis: Analyze and describe the entire image as a whole with maximum detail and completeness. F